# Preprocessing LOO Gaussian

This notebook runs the tuning phase for the Gaussian LOO experiment and saves the tuning parameters to disk. The MSE notebook can then load these parameters without recomputing `k_grid`, `m_grid`, `sqrt_m2`, or the stratified allocation.

The final MSE budget is treated separately from this preprocessing phase.

In [2]:
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd

from src.gaussian_LOO import BayesianLinearRegressionTempering
from src.normal import tune_km_grid, estimate_mom, uestimator_given_lambda
from src.startified_estimator import build_budget


## Experiment Configuration

The current coupled-MCMC implementation is still mostly written for a one-dimensional chain state, so this notebook keeps `d = 1`. The Gaussian LOO algebra in `src/gaussian_LOO.py` is written with vector-valued regression parameters in mind.

In [3]:
SEED_DATA = 2026
SEED_TUNING = 2027

n, d = 40, 1
true_theta = np.array([2.5])
sigma2_noise = 0.4**2

prior_mean = np.zeros(d)
prior_cov = np.eye(d) * 10.0

L = 20
lambda_grid = np.arange(L + 1) / L
M_lambda = 100

sigmaq = 1.0
lag = 1
nrep_km = 10
nrep_mom = 10
n_budget_pilot = 10

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)
tuning_path = output_dir / f"gaussian_loo_tuning_Mlambda{M_lambda}.pkl"


## Simulate Data

In [4]:
rng_data = np.random.default_rng(SEED_DATA)
X_data = rng_data.standard_normal((n, d))
y_data = X_data @ true_theta + rng_data.normal(0, np.sqrt(sigma2_noise), size=n)

blr_path = BayesianLinearRegressionTempering(
    X_data,
    y_data,
    prior_mean,
    prior_cov,
    sigma2_noise,
)

loo_log_densities_exact, true_value = blr_path.exact_loo_elpd()
print(f"Exact conjugate LOO ELPD: {true_value:.6f}")


Exact conjugate LOO ELPD: -26.527064


## Helper Functions

In [5]:
def make_loo_functions(index_i):
    log_target_path_i = lambda theta, path, i=index_i: blr_path.log_path(
        theta,
        path,
        i,
    )
    grad_log_target_path_i = lambda theta, i=index_i: blr_path.log_likelihood_i(theta, i)
    return log_target_path_i, grad_log_target_path_i


def make_loo_estimator(index_i, k_grid, m_grid):
    log_target_path_i, grad_log_target_path_i = make_loo_functions(index_i)
    return lambda lam: uestimator_given_lambda(
        lam,
        lambda_grid,
        k_grid,
        m_grid,
        sigmaq,
        lag,
        log_target_path_i,
        grad_log_target_path_i,
    )


## Tune All LOO Indices

For each observation index, this cell stores:

- `k_grid`, `m_grid`: coupled-chain tuning over `lambda_grid`;
- `sqrt_m2`: proposal weights for importance sampling over lambda;
- `budget`: stratified allocation for the fixed `M_lambda`.

In [6]:
np.random.seed(SEED_TUNING)

tuning_by_index = {}
summary_rows = []
start_time = time.perf_counter()

for index_i in range(n):
    print(f"Tuning LOO index {index_i + 1}/{n}")
    index_start = time.perf_counter()

    log_target_path_i, grad_log_target_path_i = make_loo_functions(index_i)

    km_grid = tune_km_grid(
        lambda_grid,
        lag=lag,
        sigmaq=sigmaq,
        log_target_path=log_target_path_i,
        nrep=nrep_km,
    )
    k_grid = km_grid["k_grid"]
    m_grid = km_grid["m_grid"]

    mom = estimate_mom(
        lambda_grid,
        k_grid,
        m_grid,
        nrep_mom,
        sigmaq,
        lag,
        log_target_path_i,
        grad_log_target_path_i,
    )
    sqrt_m2 = np.sqrt(np.maximum(mom["m2"], 1e-12))

    estimator_i = make_loo_estimator(index_i, k_grid, m_grid)
    budget = build_budget(
        L=L,
        M=M_lambda,
        estimator=estimator_i,
        n_samples_per_bin=n_budget_pilot,
    )

    elapsed = time.perf_counter() - index_start
    tuning_by_index[index_i] = {
        "k_grid": k_grid,
        "m_grid": m_grid,
        "sqrt_m2": sqrt_m2,
        "budget": budget,
        "mom_m1": mom["m1"],
        "mom_m2": mom["m2"],
        "mom_variance": mom["variance"],
        "mom_cost": mom["cost"],
        "elapsed_seconds": elapsed,
    }
    summary_rows.append({
        "index_i": index_i,
        "elapsed_seconds": elapsed,
        "mean_k": np.mean(k_grid),
        "mean_m": np.mean(m_grid),
        "budget_sum": np.sum(budget),
    })

total_elapsed = time.perf_counter() - start_time
summary_df = pd.DataFrame(summary_rows)
print(f"Total tuning time: {total_elapsed:.2f} seconds")
summary_df.head()


Tuning LOO index 1/40


Final estimation: 100%|██████████| 21/21 [00:04<00:00,  4.73it/s]


Tuning LOO index 2/40


Final estimation: 100%|██████████| 21/21 [00:04<00:00,  4.63it/s]


Tuning LOO index 3/40


...

Tuning LOO index 18/40


Final estimation: 100%|██████████| 21/21 [00:06<00:00,  3.12it/s]


Tuning LOO index 19/40


Final estimation:  14%|█▍        | 3/21 [00:00<00:05,  3.32it/s]


KeyboardInterrupt: 

## Save Tuning Artifact

In [ ]:
artifact = {
    "config": {
        "SEED_DATA": SEED_DATA,
        "SEED_TUNING": SEED_TUNING,
        "n": n,
        "d": d,
        "L": L,
        "lambda_grid": lambda_grid,
        "M_lambda": M_lambda,
        "sigmaq": sigmaq,
        "lag": lag,
        "nrep_km": nrep_km,
        "nrep_mom": nrep_mom,
        "n_budget_pilot": n_budget_pilot,
        "total_tuning_seconds": total_elapsed,
    },
    "data": {
        "X_data": X_data,
        "y_data": y_data,
        "true_theta": true_theta,
        "prior_mean": prior_mean,
        "prior_cov": prior_cov,
        "sigma2_noise": sigma2_noise,
        "loo_log_densities_exact": loo_log_densities_exact,
        "true_value": true_value,
    },
    "tuning_by_index": tuning_by_index,
    "summary": summary_df,
}

with tuning_path.open("wb") as f:
    pickle.dump(artifact, f)

print(f"Saved tuning artifact to: {tuning_path}")
